# Federal Reserve Macro Insights MVP

Minimum-viable pipeline using only free/open-source components: scrape Federal Reserve PDFs, build local retrieval, and generate a structured macro view with citations via a local LLM.

## 0) Install dependencies

In [1]:
%pip install -q requests beautifulsoup4 pypdf sentence-transformers faiss-cpu numpy pandas tqdm ollama pyarrow scikit-learn

Note: you may need to restart the kernel to use updated packages.


## 1) Imports + configuration

In [7]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timedelta, timezone
from typing import Dict, List, Any
import json
import re

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from pypdf import PdfReader
from tqdm.auto import tqdm

import faiss
from sentence_transformers import SentenceTransformer
from ollama import Client

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "fed_macro_mvp.ipynb").exists() and (PROJECT_DIR / "fed_macro_mvp").exists():
    PROJECT_DIR = PROJECT_DIR / "fed_macro_mvp"
DATA_DIR = PROJECT_DIR / "data"
RAW_PDF_DIR = DATA_DIR / "raw_pdfs"
PROCESSED_DIR = DATA_DIR / "processed"
INDEX_DIR = PROJECT_DIR / "index"
OUTPUT_DIR = PROJECT_DIR / "outputs"
for p in [RAW_PDF_DIR, PROCESSED_DIR, INDEX_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DAYS_BACK = 540
MAX_PDFS = 40
REQUEST_TIMEOUT = 20
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 8
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 180
OLLAMA_HOST = "http://127.0.0.1:11434"
# OLLAMA_MODEL = "mistral:7b-instruct"
OLLAMA_MODEL = "llama3:8b"

FOCUS_TOPICS = ["inflation", "unemployment", "labor market", "growth", "interest rates", "financial conditions", "credit"]
SEED_PAGES = [
    "https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm",
    "https://www.federalreserve.gov/monetarypolicy/fomcminutes.htm",
    "https://www.federalreserve.gov/monetarypolicy/mpr_default.htm",
    "https://www.federalreserve.gov/monetarypolicy/beigebookdefault.htm",
]

print("Project:", PROJECT_DIR)

TOP_K_TOPIC = 2
MAX_CHARS_PER_CHUNK = 700
MAX_CONTEXT_CHARS = 2800
OLLAMA_NUM_PREDICT = 320
OLLAMA_NUM_CTX = 4096
OLLAMA_TEMPERATURE = 0.0

# Investor-facing topic map for targeted retrieval
TOPIC_QUERIES = {
    "inflation": {
        "query": "What do recent Federal Reserve communications imply about inflation trend, stickiness, and upside/downside risks?",
        "keywords": ["inflation", "prices", "PCE", "CPI", "expectations"],
    },
    "unemployment": {
        "query": "What do recent communications imply about unemployment, labor demand, and job market slack?",
        "keywords": ["unemployment", "labor", "employment", "wages", "payroll"],
    },
    "growth": {
        "query": "What do recent Federal Reserve communications imply about GDP growth, demand, and activity momentum?",
        "keywords": ["growth", "GDP", "activity", "demand", "consumption", "investment"],
    },
    "policy_rates": {
        "query": "What do recent Federal Reserve communications imply about policy rate path, cuts/hikes, and reaction function?",
        "keywords": ["federal funds", "policy", "rate", "hike", "cut", "restrictive"],
    },
    "financial_conditions": {
        "query": "What do recent communications imply about financial conditions, yields, equities, dollar, and credit spreads?",
        "keywords": ["financial conditions", "yields", "equity", "dollar", "spreads", "treasury"],
    },
    "credit": {
        "query": "What do recent communications imply about bank lending standards, credit availability, and funding conditions?",
        "keywords": ["credit", "lending", "bank", "loan", "funding", "delinquency"],
    },
}


ENABLE_HYBRID_RETRIEVAL = True
ENABLE_QUERY_FUSION = True
RRF_K = 60
RECENCY_HALF_LIFE_DAYS = 180
RECENCY_BOOST = 0.35
CANDIDATES_PER_QUERY = 8

# Query variants are used for query-fusion per topic (robust, deterministic expansion)
TOPIC_QUERY_VARIANTS = {
    "inflation": [
        "inflation trend and persistence",
        "inflation expectations and upside risks",
    ],
    "unemployment": [
        "labor market slack and unemployment trajectory",
        "employment, wages, and labor demand",
    ],
    "growth": [
        "real activity and GDP momentum",
        "consumer demand and business investment outlook",
    ],
    "policy_rates": [
        "federal funds rate path and policy stance",
        "conditions for rate cuts or hikes",
    ],
    "financial_conditions": [
        "financial conditions, yields, and risk appetite",
        "treasury yields, dollar strength, and market pricing",
    ],
    "credit": [
        "bank lending standards and credit supply",
        "credit availability and funding stress",
    ],
}


ENABLE_RERANKER = False
RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
RERANK_CANDIDATE_POOL = 10


OLLAMA_MAX_RETRIES = 3
RETRY_CONTEXT_SHRINK = [1.0, 0.7, 0.45]
RETRY_PREDICT_SHRINK = [1.0, 0.75, 0.55]


Project: /Users/atheeshkrishnan/AK/DEV/hawkdove/fed_macro_mvp


## 2) Discover + download Federal Reserve PDFs

In [3]:
def _normalize_pdf_url(href: str, base_url: str) -> str | None:
    href = (href or "").strip()
    if not href:
        return None
    if href.startswith("//"):
        href = "https:" + href
    elif href.startswith("/"):
        href = "https://www.federalreserve.gov" + href
    elif not href.lower().startswith("http"):
        href = requests.compat.urljoin(base_url, href)
    if ".pdf" not in href.lower() or "federalreserve.gov" not in href.lower():
        return None
    return href.split("#")[0]


def _extract_date_hint(s: str) -> str | None:
    m = re.search(r"(20\d{2})(\d{2})(\d{2})", s)
    if m:
        y, mo, d = m.groups()
        return f"{y}-{mo}-{d}"
    y = re.search(r"(20\d{2})", s)
    return f"{y.group(1)}-01-01" if y else None


def scrape_seed_pages(seed_pages: List[str]) -> pd.DataFrame:
    rows = []
    s = requests.Session()
    s.headers.update({"User-Agent": "Mozilla/5.0 (compatible; hawkdove-mvp/1.0)"})
    for page in seed_pages:
        try:
            r = s.get(page, timeout=REQUEST_TIMEOUT)
            if r.status_code != 200:
                continue
        except Exception:
            continue
        soup = BeautifulSoup(r.text, "html.parser")
        for a in soup.find_all("a"):
            pdf_url = _normalize_pdf_url(a.get("href"), page)
            if not pdf_url:
                continue
            title = " ".join(a.get_text(" ", strip=True).split()) or Path(pdf_url).name
            rows.append({"source": "seed_page", "source_page": page, "pdf_url": pdf_url, "title": title, "date_hint": _extract_date_hint(pdf_url)})
    if not rows:
        return pd.DataFrame(columns=["source", "source_page", "pdf_url", "title", "date_hint"])
    return pd.DataFrame(rows).drop_duplicates(subset=["pdf_url"]).reset_index(drop=True)


def probe_known_patterns(days_back: int = DAYS_BACK) -> pd.DataFrame:
    base = "https://www.federalreserve.gov/monetarypolicy/files/"
    s = requests.Session()
    s.headers.update({"User-Agent": "Mozilla/5.0 (compatible; hawkdove-mvp/1.0)"})
    rows = []
    today = datetime.now(timezone.utc).date()
    for offset in tqdm(range(days_back), desc="Probing patterns"):
        dt = today - timedelta(days=offset)
        ymd = dt.strftime("%Y%m%d")
        for fname in [f"fomcminutes{ymd}.pdf", f"monetary{ymd}a1.pdf"]:
            url = base + fname
            try:
                r = s.head(url, allow_redirects=True, timeout=8)
            except Exception:
                continue
            ctype = (r.headers.get("Content-Type") or "").lower()
            if r.status_code == 200 and ("pdf" in ctype or url.endswith(".pdf")):
                rows.append({"source": "known_pattern", "source_page": "monetarypolicy/files", "pdf_url": url, "title": fname, "date_hint": dt.isoformat()})
    if not rows:
        return pd.DataFrame(columns=["source", "source_page", "pdf_url", "title", "date_hint"])
    return pd.DataFrame(rows).drop_duplicates(subset=["pdf_url"]).reset_index(drop=True)


def build_catalog(seed_pages: List[str], days_back: int = DAYS_BACK) -> pd.DataFrame:
    a = scrape_seed_pages(seed_pages)
    b = probe_known_patterns(days_back)
    c = pd.concat([a, b], ignore_index=True).drop_duplicates(subset=["pdf_url"])
    if c.empty:
        return c
    c["parsed_date"] = pd.to_datetime(c["date_hint"], errors="coerce")
    return c.sort_values("parsed_date", ascending=False, na_position="last").reset_index(drop=True)


def download_catalog(catalog_df: pd.DataFrame, out_dir: Path, max_pdfs: int = MAX_PDFS) -> pd.DataFrame:
    if catalog_df.empty:
        return catalog_df
    s = requests.Session()
    s.headers.update({"User-Agent": "Mozilla/5.0 (compatible; hawkdove-mvp/1.0)"})
    rows = []
    for _, row in catalog_df.head(max_pdfs).iterrows():
        url = row["pdf_url"]
        fname = Path(url).name
        local = out_dir / fname
        if local.exists() and local.stat().st_size > 0:
            rows.append({**row.to_dict(), "local_path": str(local), "status": "exists"})
            continue
        try:
            r = s.get(url, timeout=REQUEST_TIMEOUT)
            if r.status_code == 200 and r.content:
                local.write_bytes(r.content)
                rows.append({**row.to_dict(), "local_path": str(local), "status": "downloaded"})
            else:
                rows.append({**row.to_dict(), "local_path": str(local), "status": f"http_{r.status_code}"})
        except Exception as e:
            rows.append({**row.to_dict(), "local_path": str(local), "status": f"error:{e}"})
    return pd.DataFrame(rows)


catalog_df = build_catalog(SEED_PAGES, DAYS_BACK)
print("Catalog candidates:", len(catalog_df))
if not catalog_df.empty:
    display(catalog_df.head(20))

download_df = download_catalog(catalog_df, RAW_PDF_DIR, MAX_PDFS)
print("Downloaded/available:", len(download_df))
if not download_df.empty:
    display(download_df[["title", "status", "local_path"]].head(20))

manifest_path = PROCESSED_DIR / "download_manifest.csv"
if not download_df.empty:
    download_df.to_csv(manifest_path, index=False)
    print("Saved:", manifest_path)


Probing patterns: 100%|██████████| 540/540 [01:07<00:00,  8.00it/s]

Catalog candidates: 161


,source,source_page,pdf_url,title,date_hint,parsed_date
0,seed_page,https://www.federalreserve.gov/monetarypolicy/...,https://www.federalreserve.gov/monetarypolicy/...,PDF,2026-01-28,2026-01-28
1,seed_page,https://www.federalreserve.gov/monetarypolicy/...,https://www.federalreserve.gov/monetarypolicy/...,PDF,2026-01-28,2026-01-28
2,seed_page,https://www.federalreserve.gov/monetarypolicy/...,https://www.federalreserve.gov/monetarypolicy/...,PDF,2025-12-10,2025-12-10
3,seed_page,https://www.federalreserve.gov/monetarypolicy/...,https://www.federalreserve.gov/monetarypolicy/...,PDF,2025-12-10,2025-12-10
4,seed_page,https://www.federalreserve.gov/monetarypolicy/...,https://www.federalreserve.gov/monetarypolicy/...,PDF,2025-12-10,2025-12-10
5,seed_page,https://www.federalreserve.gov/monetarypolicy/...,https://www.federalreserve.gov/monetarypolicy/...,PDF,2025-10-29,2025-10-29
6,seed_page,https://www.federalreserve.gov/monetarypolicy/...,https://www.federalreserve.gov/monetarypolicy/...,PDF,2025-10-29,2025-10-29
7,seed_page,https://www.federalreserve.gov/monetarypolicy/...,https://www.federalreserve.gov/monetarypolicy/...,PDF,2025-09-17,2025-09-17
8,seed_page,https://www.federalreserve.gov/monetarypolicy/...,https://www.federalreserve.gov/monetarypolicy/...,PDF,2025-09-17,2025-09-17
9,seed_page,https://www.federalreserve.gov/monetarypolicy/...,https://www.federalreserve.gov/monetarypolicy/...,PDF,2025-09-17,2025-09-17


Downloaded/available: 40


,title,status,local_path
0,PDF,exists,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
1,PDF,exists,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
2,PDF,exists,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
3,PDF,exists,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
4,PDF,exists,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
5,PDF,exists,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
6,PDF,exists,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
7,PDF,exists,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
8,PDF,exists,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...
9,PDF,exists,/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_mac...


Saved: /Users/atheeshkrishnan/AK/DEV/hawkdove/fed_macro_mvp/data/processed/download_manifest.csv


## 3) Parse + chunk + index

In [4]:
def read_pdf_text(path: Path) -> str:
    try:
        reader = PdfReader(str(path))
    except Exception:
        return ""
    pages = []
    for p in reader.pages:
        try:
            pages.append(p.extract_text() or "")
        except Exception:
            pages.append("")
    text = "\n".join(pages)
    return re.sub(r"\s+", " ", text).strip()


def chunk_text(text: str, size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    if not text:
        return []
    out = []
    step = max(1, size - overlap)
    i = 0
    while i < len(text):
        j = min(len(text), i + size)
        out.append(text[i:j])
        if j == len(text):
            break
        i += step
    return out


def get_topic_flags(text: str) -> List[str]:
    t = text.lower()
    return [x for x in FOCUS_TOPICS if x in t]


def build_chunks(download_manifest: pd.DataFrame) -> pd.DataFrame:
    rows = []
    if download_manifest.empty:
        return pd.DataFrame()
    ok = download_manifest[download_manifest["status"].isin(["downloaded", "exists"])].copy()
    for _, row in tqdm(ok.iterrows(), total=len(ok), desc="Parsing PDFs"):
        f = Path(row["local_path"])
        if not f.exists():
            continue
        text = read_pdf_text(f)
        if len(text) < 200:
            continue
        parts = chunk_text(text)
        for idx, part in enumerate(parts):
            if len(part.strip()) < 80:
                continue
            rows.append({
                "chunk_id": f"{f.name}::chunk{idx:04d}",
                "doc_id": f.name,
                "pdf_url": row.get("pdf_url", ""),
                "date_hint": row.get("date_hint", ""),
                "chunk_index": idx,
                "text": part,
                "topic_flags": get_topic_flags(part),
                "text_len": len(part),
            })
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).drop_duplicates(subset=["chunk_id"]).reset_index(drop=True)


def norm_rows(x: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(x, axis=1, keepdims=True)
    n[n == 0] = 1.0
    return x / n


def save_table(df: pd.DataFrame, preferred_path: Path) -> Path:
    try:
        df.to_parquet(preferred_path, index=False)
        return preferred_path
    except Exception as e:
        fallback = preferred_path.with_suffix(".csv")
        df.to_csv(fallback, index=False)
        print(f"[warn] Parquet unavailable ({e}); saved CSV fallback: {fallback}")
        return fallback


def build_faiss_index(chunks_df: pd.DataFrame, embed_model_name: str = EMBED_MODEL_NAME) -> Dict[str, Any]:
    if chunks_df.empty:
        raise ValueError("No chunks. Run ingestion first.")
    model = SentenceTransformer(embed_model_name)
    vecs = model.encode(chunks_df["text"].tolist(), show_progress_bar=True, convert_to_numpy=True).astype("float32")
    vecs = norm_rows(vecs)
    dim = vecs.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(vecs)

    index_path = INDEX_DIR / "fed_chunks.index"
    meta_path = save_table(chunks_df, INDEX_DIR / "fed_chunks_meta.parquet")
    cfg_path = INDEX_DIR / "index_config.json"

    faiss.write_index(index, str(index_path))
    

    cfg = {
        "embed_model": embed_model_name,
        "dimension": dim,
        "rows": int(len(chunks_df)),
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "index_path": str(index_path),
        "meta_path": str(meta_path),
    }
    with open(cfg_path, "w") as f:
        json.dump(cfg, f, indent=2)
    return cfg


chunks_df = build_chunks(download_df)
print("Chunk rows:", len(chunks_df))
if not chunks_df.empty:
    display(chunks_df.head(5))
    chunks_path = save_table(chunks_df, PROCESSED_DIR / "chunks.parquet")
    print("Saved:", chunks_path)

if not chunks_df.empty:
    cfg = build_faiss_index(chunks_df, EMBED_MODEL_NAME)
    print(json.dumps(cfg, indent=2))


Parsing PDFs: 100%|██████████| 40/40 [00:30<00:00,  1.31it/s]

Chunk rows: 1551


,chunk_id,doc_id,pdf_url,date_hint,chunk_index,text,topic_flags,text_len
0,monetary20260128a1.pdf::chunk0000,monetary20260128a1.pdf,https://www.federalreserve.gov/monetarypolicy/...,2026-01-28,0,"For release at 2:00 p.m. EST January 28, 2026 ...","[inflation, unemployment]",1200
1,monetary20260128a1.pdf::chunk0001,monetary20260128a1.pdf,https://www.federalreserve.gov/monetarypolicy/...,2026-01-28,1,"cy, the Committee will continue to monitor the...","[inflation, labor market]",1200
2,monetary20260128a1.pdf::chunk0002,monetary20260128a1.pdf,https://www.federalreserve.gov/monetarypolicy/...,2026-01-28,2,"at 2:00 p.m. EST January 28, 2026 Decisions R...",[],1200
3,monetary20260128a1.pdf::chunk0003,monetary20260128a1.pdf,https://www.federalreserve.gov/monetarypolicy/...,2026-01-28,3,percent. o Conduct standing overnight reverse ...,[credit],1178
4,fomcminutes20260128.pdf::chunk0000,fomcminutes20260128.pdf,https://www.federalreserve.gov/monetarypolicy/...,2026-01-28,0,FOMC FEDERAL RESERVE SYSTEM Minutes of the Fed...,[],1200


Saved: /Users/atheeshkrishnan/AK/DEV/hawkdove/fed_macro_mvp/data/processed/chunks.parquet


Batches: 100%|██████████| 49/49 [01:17<00:00,  1.57s/it]

{
  "embed_model": "sentence-transformers/all-MiniLM-L6-v2",
  "dimension": 384,
  "rows": 1551,
  "created_utc": "2026-03-16T00:30:01.979724+00:00",
  "index_path": "/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_macro_mvp/index/fed_chunks.index",
  "meta_path": "/Users/atheeshkrishnan/AK/DEV/hawkdove/fed_macro_mvp/index/fed_chunks_meta.parquet"
}


## 4) Investor-Focused Retrieval + Local LLM Synthesis

This section is optimized for speed and investor relevance:
- Topic-targeted retrieval (inflation, unemployment, growth, policy rates, financial conditions, credit)
- Context compaction to reduce LLM token load
- Schema-constrained JSON output for reliable downstream use
- Built-in latency and evidence-quality checks


In [8]:
import time

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    HAVE_SKLEARN = True
except Exception:
    HAVE_SKLEARN = False


def load_bundle(index_dir: Path, embed_model_name: str = EMBED_MODEL_NAME):
    idx = faiss.read_index(str(index_dir / "fed_chunks.index"))
    meta_parquet = index_dir / "fed_chunks_meta.parquet"
    meta_csv = index_dir / "fed_chunks_meta.csv"
    if meta_parquet.exists():
        meta = pd.read_parquet(meta_parquet)
    elif meta_csv.exists():
        meta = pd.read_csv(meta_csv)
    else:
        raise FileNotFoundError("No index metadata found (expected fed_chunks_meta.parquet or fed_chunks_meta.csv)")
    em = SentenceTransformer(embed_model_name)
    return idx, meta, em


def check_ollama_ready(host: str = OLLAMA_HOST, model: str = OLLAMA_MODEL) -> None:
    try:
        r = requests.get(f"{host}/api/tags", timeout=3)
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        raise RuntimeError(
            "Ollama is not reachable. Start Ollama first (e.g., `ollama serve` or open the Ollama app). "
            f"Host tried: {host}. Original error: {e}"
        )

    names = {m.get("name", "") for m in data.get("models", []) if isinstance(m, dict)}
    if model not in names:
        hint = ", ".join(sorted(names)) if names else "<no models found>"
        raise RuntimeError(
            f"Model '{model}' not found in Ollama. Available: {hint}. "
            f"Run: ollama pull {model} (or update OLLAMA_MODEL)."
        )


def load_reranker(enabled: bool = ENABLE_RERANKER, model_name: str = RERANK_MODEL_NAME):
    if not enabled:
        return None
    try:
        from sentence_transformers import CrossEncoder
        return CrossEncoder(model_name, max_length=512)
    except Exception as e:
        print(f"[warn] Reranker disabled ({e})")
        return None


def recency_score(date_hint: Any, half_life_days: int = RECENCY_HALF_LIFE_DAYS) -> float:
    if not date_hint or pd.isna(date_hint):
        return 0.5
    ts = pd.to_datetime(date_hint, errors="coerce", utc=True)
    if pd.isna(ts):
        return 0.5
    age_days = max(0.0, (pd.Timestamp.now(tz="UTC") - ts).total_seconds() / 86400.0)
    return float(np.exp(-np.log(2) * age_days / max(1, half_life_days)))


def minmax_scale(values: pd.Series) -> pd.Series:
    v = values.astype(float)
    lo, hi = float(v.min()), float(v.max())
    if hi - lo < 1e-12:
        return pd.Series([1.0] * len(v), index=v.index)
    return (v - lo) / (hi - lo)


def dense_retrieve(query: str, idx, meta_df: pd.DataFrame, emb_model, top_k: int) -> pd.DataFrame:
    q = emb_model.encode([query], convert_to_numpy=True).astype("float32")
    q = norm_rows(q)
    scores, ids = idx.search(q, top_k)
    out = []
    for score, i in zip(scores[0], ids[0]):
        if i < 0 or i >= len(meta_df):
            continue
        out.append({"chunk_id": meta_df.iloc[int(i)]["chunk_id"], "score": float(score)})
    if not out:
        return pd.DataFrame(columns=["chunk_id", "score"])
    return pd.DataFrame(out).drop_duplicates(subset=["chunk_id"]).sort_values("score", ascending=False).reset_index(drop=True)


def build_sparse_index(meta_df: pd.DataFrame):
    if not HAVE_SKLEARN:
        return None
    try:
        vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1, max_df=0.95)
        mat = vec.fit_transform(meta_df["text"].fillna("").astype(str).tolist())
        return vec, mat
    except Exception as e:
        print(f"[warn] Sparse index disabled: {e}")
        return None


def sparse_retrieve(query: str, sparse_bundle, meta_df: pd.DataFrame, top_k: int) -> pd.DataFrame:
    if sparse_bundle is None:
        return pd.DataFrame(columns=["chunk_id", "score"])
    vec, mat = sparse_bundle
    try:
        qv = vec.transform([query])
        sims = (qv @ mat.T).toarray().ravel()
    except Exception:
        return pd.DataFrame(columns=["chunk_id", "score"])

    if sims.size == 0:
        return pd.DataFrame(columns=["chunk_id", "score"])

    k = min(top_k, sims.size)
    idxs = np.argpartition(sims, -k)[-k:]
    ranked = idxs[np.argsort(sims[idxs])[::-1]]
    out = [{"chunk_id": meta_df.iloc[int(i)]["chunk_id"], "score": float(sims[int(i)])} for i in ranked]
    if not out:
        return pd.DataFrame(columns=["chunk_id", "score"])
    return pd.DataFrame(out).drop_duplicates(subset=["chunk_id"]).sort_values("score", ascending=False).reset_index(drop=True)


def rrf_fuse(rank_frames: list[pd.DataFrame], score_col: str = "score", k: int = RRF_K) -> pd.DataFrame:
    agg = {}
    for df in rank_frames:
        if df is None or df.empty:
            continue
        ordered = df.sort_values(score_col, ascending=False).reset_index(drop=True)
        for rank, cid in enumerate(ordered["chunk_id"].tolist(), start=1):
            agg[cid] = agg.get(cid, 0.0) + 1.0 / (k + rank)
    if not agg:
        return pd.DataFrame(columns=["chunk_id", "fusion_score"])
    return pd.DataFrame({"chunk_id": list(agg.keys()), "fusion_score": list(agg.values())}).sort_values("fusion_score", ascending=False).reset_index(drop=True)


def build_topic_queries(topic: str, cfg: Dict[str, Any]) -> list[str]:
    queries = [cfg["query"]]
    if ENABLE_QUERY_FUSION:
        for qv in TOPIC_QUERY_VARIANTS.get(topic, []):
            queries.append(f"Federal Reserve {qv}")
    return list(dict.fromkeys(queries))


def apply_reranker(query: str, candidates: pd.DataFrame, reranker) -> pd.DataFrame:
    if reranker is None or candidates.empty:
        return candidates
    work = candidates.head(RERANK_CANDIDATE_POOL).copy()
    pairs = [[query, str(t)[:1200]] for t in work["text"].fillna("").tolist()]
    try:
        scores = reranker.predict(pairs)
    except Exception as e:
        print(f"[warn] rerank predict failed: {e}")
        return candidates
    work["rerank_score"] = [float(x) for x in scores]
    out = candidates.merge(work[["chunk_id", "rerank_score"]], on="chunk_id", how="left")
    out["rerank_score"] = out["rerank_score"].fillna(out["score"])
    return out


def retrieve_topic_hybrid(topic: str, cfg: Dict[str, Any], idx, meta_df: pd.DataFrame, emb_model, sparse_bundle, reranker=None) -> pd.DataFrame:
    query_fused = []
    topic_queries = build_topic_queries(topic, cfg)

    for q in topic_queries:
        dense_df = dense_retrieve(q, idx, meta_df, emb_model, top_k=CANDIDATES_PER_QUERY)
        if ENABLE_HYBRID_RETRIEVAL:
            sparse_df = sparse_retrieve(q, sparse_bundle, meta_df, top_k=CANDIDATES_PER_QUERY)
            fused_q = rrf_fuse([dense_df, sparse_df], score_col="score", k=RRF_K)
        else:
            fused_q = rrf_fuse([dense_df], score_col="score", k=RRF_K)
        if not fused_q.empty:
            query_fused.append(fused_q.rename(columns={"fusion_score": "score"}))

    merged = rrf_fuse(query_fused, score_col="score", k=RRF_K)
    if merged.empty:
        return pd.DataFrame(columns=["chunk_id", "doc_id", "date_hint", "topic_flags", "text", "score", "rerank_score", "final_score", "recency"])

    mlookup = meta_df.drop_duplicates(subset=["chunk_id"]).set_index("chunk_id")
    rows = []
    for _, r in merged.head(CANDIDATES_PER_QUERY).iterrows():
        cid = r["chunk_id"]
        if cid not in mlookup.index:
            continue
        meta = mlookup.loc[cid]
        rows.append({
            "chunk_id": cid,
            "doc_id": meta.get("doc_id", ""),
            "date_hint": meta.get("date_hint", ""),
            "topic_flags": meta.get("topic_flags", []),
            "text": meta.get("text", ""),
            "score": float(r["fusion_score"]),
        })

    if not rows:
        return pd.DataFrame(columns=["chunk_id", "doc_id", "date_hint", "topic_flags", "text", "score", "rerank_score", "final_score", "recency"])

    out = pd.DataFrame(rows)
    out = apply_reranker(cfg["query"], out, reranker)
    base_col = "rerank_score" if "rerank_score" in out.columns else "score"
    out["base_score_norm"] = minmax_scale(out[base_col])
    out["recency"] = out["date_hint"].apply(lambda x: recency_score(x, RECENCY_HALF_LIFE_DAYS))
    out["final_score"] = out["base_score_norm"] * ((1.0 - RECENCY_BOOST) + RECENCY_BOOST * out["recency"])
    return out.sort_values("final_score", ascending=False).reset_index(drop=True).head(TOP_K_TOPIC)


def split_sentences(text: str) -> list[str]:
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", str(text)) if s.strip()]


def topic_snippet(text: str, keywords: list[str], max_chars: int = MAX_CHARS_PER_CHUNK) -> str:
    sentences = split_sentences(text)
    if not sentences:
        return str(text)[:max_chars]
    chosen = []
    lower_keys = [k.lower() for k in keywords]
    for s in sentences:
        sl = s.lower()
        if any(k in sl for k in lower_keys):
            chosen.append(s)
        if len(" ".join(chosen)) >= max_chars:
            break
    if not chosen:
        chosen = sentences[:3]
    return " ".join(chosen)[:max_chars]


def build_topic_context(topic: str, hits_df: pd.DataFrame, keywords: list[str], top_n: int = TOP_K_TOPIC) -> tuple[str, list[str]]:
    blocks, used_ids = [], []
    if hits_df.empty:
        return "", used_ids
    for _, r in hits_df.head(top_n).iterrows():
        snippet = topic_snippet(r["text"], keywords, max_chars=MAX_CHARS_PER_CHUNK)
        blocks.append(f"[topic={topic}; chunk_id={r['chunk_id']}; doc_id={r['doc_id']}; score={r['final_score']:.4f}; recency={r['recency']:.3f}]\n{snippet}")
        used_ids.append(r["chunk_id"])
    return "\n\n".join(blocks), used_ids


def retrieve_multi_topic(idx, meta_df: pd.DataFrame, emb_model, sparse_bundle, reranker=None) -> dict[str, pd.DataFrame]:
    topic_hits = {}
    for topic, cfg in TOPIC_QUERIES.items():
        topic_hits[topic] = retrieve_topic_hybrid(topic, cfg, idx, meta_df, emb_model, sparse_bundle, reranker=reranker)
    return topic_hits


def build_investor_context(topic_hits: dict[str, pd.DataFrame]) -> tuple[str, list[str]]:
    sections, all_ids = [], []
    total_chars = 0
    for topic, cfg in TOPIC_QUERIES.items():
        section_text, ids = build_topic_context(topic, topic_hits.get(topic, pd.DataFrame()), cfg["keywords"], TOP_K_TOPIC)
        if not section_text:
            continue
        block = f"## {topic}\n{section_text}"
        if total_chars + len(block) > MAX_CONTEXT_CHARS:
            break
        sections.append(block)
        total_chars += len(block)
        all_ids.extend(ids)
    return "\n\n".join(sections), list(dict.fromkeys(all_ids))


INVESTOR_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "generated_at_utc": {"type": "string", "minLength": 1},
        "executive_summary": {"type": "string", "minLength": 1, "maxLength": 420},
        "regime_call": {
            "type": "object",
            "properties": {
                "growth_momentum": {"type": "string", "minLength": 1, "maxLength": 80},
                "inflation_trend": {"type": "string", "minLength": 1, "maxLength": 120},
                "policy_bias": {"type": "string", "minLength": 1, "maxLength": 120},
                "recession_risk": {"type": "string", "minLength": 1, "maxLength": 120},
                "confidence": {"type": "string", "minLength": 1, "maxLength": 40}
            },
            "required": ["growth_momentum", "inflation_trend", "policy_bias", "recession_risk", "confidence"]
        },
        "topic_signals": {
            "type": "array",
            "minItems": 6,
            "maxItems": 6,
            "items": {
                "type": "object",
                "properties": {
                    "topic": {"type": "string", "minLength": 1},
                    "view": {"type": "string", "minLength": 1, "maxLength": 180},
                    "why_it_matters": {"type": "string", "minLength": 1, "maxLength": 200},
                    "confidence": {"type": "string", "minLength": 1, "maxLength": 40},
                    "evidence": {"type": "array", "maxItems": 3, "items": {"type": "string"}}
                },
                "required": ["topic", "view", "why_it_matters", "confidence", "evidence"]
            }
        },
        "investor_takeaways": {
            "type": "array",
            "minItems": 1,
            "maxItems": 2,
            "items": {
                "type": "object",
                "properties": {
                    "horizon": {"type": "string", "minLength": 1, "maxLength": 80},
                    "thesis": {"type": "string", "minLength": 1, "maxLength": 220},
                    "implications_for_risk_assets": {"type": "string", "minLength": 1, "maxLength": 220},
                    "monitoring_triggers": {"type": "array", "maxItems": 4, "items": {"type": "string", "maxLength": 80}},
                    "evidence": {"type": "array", "maxItems": 4, "items": {"type": "string"}}
                },
                "required": ["horizon", "thesis", "implications_for_risk_assets", "monitoring_triggers", "evidence"]
            }
        },
        "citations": {
            "type": "array",
            "maxItems": 10,
            "items": {
                "type": "object",
                "properties": {
                    "chunk_id": {"type": "string", "minLength": 1},
                    "doc_id": {"type": "string", "minLength": 1},
                    "why_relevant": {"type": "string", "minLength": 1, "maxLength": 140}
                },
                "required": ["chunk_id", "doc_id", "why_relevant"]
            }
        }
    },
    "required": ["generated_at_utc", "executive_summary", "regime_call", "topic_signals", "investor_takeaways", "citations"]
}


def generate_investor_view(question: str, context: str, model: str = OLLAMA_MODEL, num_predict: int | None = None) -> str:
    if num_predict is None:
        num_predict = OLLAMA_NUM_PREDICT

    system = " ".join([
        "You are a macro strategist writing an investor-grade U.S. macro brief from Federal Reserve texts.",
        "Use only provided evidence context and be explicit about uncertainty.",
        "Return STRICT JSON only, with no text before or after JSON.",
        "Keep output concise and dense: short phrases, no long prose.",
        "Required top-level keys: generated_at_utc, executive_summary, regime_call, topic_signals, investor_takeaways, citations.",
        "Include exactly one topic_signals entry for each topic: inflation, unemployment, growth, policy_rates, financial_conditions, credit.",
        "For every evidence field, values must be chunk_id strings present in context.",
        "Do not invent chunk IDs.",
    ])

    user = f"Question: {question}\n\nContext:\n{context}"
    client = Client(host=OLLAMA_HOST)
    out = client.chat(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        options={
            "temperature": OLLAMA_TEMPERATURE,
            "num_predict": int(num_predict),
            "num_ctx": OLLAMA_NUM_CTX,
        },
        format=INVESTOR_JSON_SCHEMA,
    )
    return out["message"]["content"]


def parse_json(text: str) -> Dict[str, Any] | None:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text).strip()
        text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.S)
        if not m:
            return None
        try:
            return json.loads(m.group(0))
        except Exception:
            return None


def validate_investor_json(parsed: Dict[str, Any], valid_ids: set[str]) -> Dict[str, Any]:
    required_top = ["generated_at_utc", "executive_summary", "regime_call", "topic_signals", "investor_takeaways", "citations"]
    report = {
        "missing_top_keys": [k for k in required_top if k not in parsed],
        "bad_shape": [],
        "bad_evidence_ids": [],
        "unknown_citation_ids": [],
        "missing_topics": [],
        "extra_topics": [],
    }

    expected_topics = set(TOPIC_QUERIES.keys())
    if not isinstance(parsed.get("topic_signals"), list):
        report["bad_shape"].append("topic_signals")
    else:
        found_topics = []
        for i, x in enumerate(parsed["topic_signals"]):
            if not isinstance(x, dict):
                report["bad_shape"].append(f"topic_signals[{i}]")
                continue
            t = x.get("topic")
            if isinstance(t, str):
                found_topics.append(t)
            ev = x.get("evidence")
            if not isinstance(ev, list):
                report["bad_shape"].append(f"topic_signals[{i}].evidence")
            else:
                for e in ev:
                    if e not in valid_ids:
                        report["bad_evidence_ids"].append({"section": f"topic_signals[{i}]", "value": e})

        found_set = set(found_topics)
        report["missing_topics"] = sorted(expected_topics - found_set)
        report["extra_topics"] = sorted(found_set - expected_topics)

    if not isinstance(parsed.get("investor_takeaways"), list):
        report["bad_shape"].append("investor_takeaways")
    else:
        for i, x in enumerate(parsed["investor_takeaways"]):
            if not isinstance(x, dict):
                report["bad_shape"].append(f"investor_takeaways[{i}]")
                continue
            ev = x.get("evidence")
            if not isinstance(ev, list):
                report["bad_shape"].append(f"investor_takeaways[{i}].evidence")
            else:
                for e in ev:
                    if e not in valid_ids:
                        report["bad_evidence_ids"].append({"section": f"investor_takeaways[{i}]", "value": e})

    citations = parsed.get("citations")
    if not isinstance(citations, list):
        report["bad_shape"].append("citations")
    else:
        for i, c in enumerate(citations):
            if not isinstance(c, dict) or "chunk_id" not in c:
                report["bad_shape"].append(f"citations[{i}]")
                continue
            if c["chunk_id"] not in valid_ids:
                report["unknown_citation_ids"].append(c["chunk_id"])

    return report


def quality_ok(report: Dict[str, Any]) -> bool:
    return (
        len(report["missing_top_keys"]) == 0
        and len(report["bad_shape"]) == 0
        and len(report["bad_evidence_ids"]) == 0
        and len(report["unknown_citation_ids"]) == 0
        and len(report["missing_topics"]) == 0
    )


def generation_retry_plan(max_context_chars: int, num_predict: int, max_retries: int = OLLAMA_MAX_RETRIES):
    plan = []
    for i in range(max_retries):
        ctx_factor = RETRY_CONTEXT_SHRINK[min(i, len(RETRY_CONTEXT_SHRINK) - 1)]
        pred_factor = RETRY_PREDICT_SHRINK[min(i, len(RETRY_PREDICT_SHRINK) - 1)]
        plan.append({
            "attempt": i + 1,
            "context_chars": max(1500, int(max_context_chars * ctx_factor)),
            "num_predict": max(180, int(num_predict * pred_factor)),
        })
    # dedupe identical retries
    unique = []
    seen = set()
    for x in plan:
        key = (x["context_chars"], x["num_predict"])
        if key not in seen:
            unique.append(x)
            seen.add(key)
    return unique


def run_generation_with_retries(question: str, context: str, model: str, valid_ids: set[str]):
    attempts = generation_retry_plan(MAX_CONTEXT_CHARS, OLLAMA_NUM_PREDICT, OLLAMA_MAX_RETRIES)
    last_text, last_parsed, last_quality = "", None, None
    logs = []

    for a in attempts:
        ctx = context[:a["context_chars"]]
        t0 = time.time()
        text = generate_investor_view(question, ctx, model=model, num_predict=a["num_predict"])
        lat = time.time() - t0

        parsed_obj = parse_json(text)
        if parsed_obj is None:
            logs.append({"attempt": a["attempt"], "status": "parse_failed", "latency_s": round(lat, 2), "context_chars": a["context_chars"], "num_predict": a["num_predict"]})
            last_text = text
            continue

        # backfill timestamp if model returns blank despite schema
        if not str(parsed_obj.get("generated_at_utc", "")).strip():
            parsed_obj["generated_at_utc"] = datetime.now(timezone.utc).isoformat()

        q = validate_investor_json(parsed_obj, valid_ids)
        ok = quality_ok(q)
        logs.append({"attempt": a["attempt"], "status": "ok" if ok else "quality_failed", "latency_s": round(lat, 2), "context_chars": a["context_chars"], "num_predict": a["num_predict"]})
        last_text, last_parsed, last_quality = text, parsed_obj, q
        if ok:
            return text, parsed_obj, q, pd.DataFrame(logs)

    return last_text, last_parsed, last_quality, pd.DataFrame(logs)


index, meta_df, emb_model = load_bundle(INDEX_DIR, EMBED_MODEL_NAME)
query = "Build an investor-focused 6-12 month U.S. macro view from recent Federal Reserve communications."

sparse_t0 = time.time()
sparse_bundle = build_sparse_index(meta_df)
sparse_latency = time.time() - sparse_t0
if sparse_bundle is None:
    print("Sparse retrieval: disabled (sklearn unavailable or index build failed)")
else:
    print(f"Sparse retrieval index built in {sparse_latency:.2f}s")

rerank_t0 = time.time()
reranker = load_reranker(ENABLE_RERANKER, RERANK_MODEL_NAME)
rerank_latency = time.time() - rerank_t0
print(f"Reranker status: {'enabled' if reranker is not None else 'disabled'} (load time: {rerank_latency:.2f}s)")

retrieval_t0 = time.time()
topic_hits = retrieve_multi_topic(index, meta_df, emb_model, sparse_bundle, reranker=reranker)
retrieval_latency = time.time() - retrieval_t0

summary_rows = []
for topic, h in topic_hits.items():
    summary_rows.append({
        "topic": topic,
        "hits": int(len(h)),
        "top_final_score": float(h["final_score"].iloc[0]) if len(h) else None,
        "top_chunk_id": h["chunk_id"].iloc[0] if len(h) else None,
    })

topic_summary_df = pd.DataFrame(summary_rows)
print(f"Topic retrieval completed in {retrieval_latency:.2f}s")
display(topic_summary_df)

context, context_chunk_ids = build_investor_context(topic_hits)
print(f"Context size: {len(context)} chars from {len(context_chunk_ids)} chunks")

flat_hits = [h.assign(topic=topic) for topic, h in topic_hits.items() if not h.empty]
if flat_hits:
    hits_df = pd.concat(flat_hits, ignore_index=True)
    hits_df = hits_df.sort_values("final_score", ascending=False).drop_duplicates(subset=["chunk_id"]).reset_index(drop=True)
    display(hits_df[["topic", "chunk_id", "doc_id", "final_score", "recency"]].head(12))
else:
    hits_df = pd.DataFrame(columns=["topic", "chunk_id", "doc_id", "date_hint", "score", "recency", "final_score", "topic_flags", "text"])

check_ollama_ready(OLLAMA_HOST, OLLAMA_MODEL)
llm_t0 = time.time()
llm_text, parsed, quality, attempt_log = run_generation_with_retries(query, context, OLLAMA_MODEL, set(context_chunk_ids))
llm_latency = time.time() - llm_t0
print(f"LLM stage completed in {llm_latency:.2f}s")
if not attempt_log.empty:
    display(attempt_log)

print(llm_text)

if parsed is None:
    print("[check] JSON parse failed after retries")
else:
    print("[check] Missing top keys:", quality["missing_top_keys"] if quality["missing_top_keys"] else "None")
    print("[check] Bad shape:", quality["bad_shape"] if quality["bad_shape"] else "None")
    print("[check] Bad evidence IDs:", quality["bad_evidence_ids"] if quality["bad_evidence_ids"] else "None")
    print("[check] Unknown citation IDs:", quality["unknown_citation_ids"] if quality["unknown_citation_ids"] else "None")
    print("[check] Missing topics:", quality["missing_topics"] if quality["missing_topics"] else "None")
    print("[check] Extra topics:", quality["extra_topics"] if quality["extra_topics"] else "None")


Sparse retrieval index built in 0.33s
Reranker status: disabled (load time: 0.00s)
Topic retrieval completed in 0.72s


,topic,hits,top_final_score,top_chunk_id
0,inflation,2,0.702222,fomcminutes20241107.pdf::chunk0011
1,unemployment,2,0.824978,fomcminutes20250917.pdf::chunk0020
2,growth,2,0.724424,20250207_mprfullreport.pdf::chunk0175
3,policy_rates,2,0.724424,20250207_mprfullreport.pdf::chunk0175
4,financial_conditions,2,0.693076,fomcminutes20240918.pdf::chunk0000
5,credit,2,0.702222,fomcminutes20241107.pdf::chunk0016


Context size: 2742 chars from 4 chunks


,topic,chunk_id,doc_id,final_score,recency
0,unemployment,fomcminutes20250917.pdf::chunk0020,fomcminutes20250917.pdf,0.824978,0.499937
1,growth,20250207_mprfullreport.pdf::chunk0175,20250207_mprfullreport.pdf,0.724424,0.212640
2,inflation,fomcminutes20241107.pdf::chunk0011,fomcminutes20241107.pdf,0.702222,0.149205
3,credit,fomcminutes20241107.pdf::chunk0016,fomcminutes20241107.pdf,0.702222,0.149205
4,financial_conditions,fomcminutes20240918.pdf::chunk0000,fomcminutes20240918.pdf,0.693076,0.123074
5,financial_conditions,20250620_mprfullreport.pdf::chunk0013,20250620_mprfullreport.pdf,0.558442,0.354872
6,credit,fomcminutes20260128.pdf::chunk0016,fomcminutes20260128.pdf,0.191922,0.834339
7,growth,fomcprojtabl20250618.pdf::chunk0001,fomcprojtabl20250618.pdf,0.173441,0.352150
8,policy_rates,fomcminutes20250507.pdf::chunk0008,fomcminutes20250507.pdf,0.150527,0.299562
9,unemployment,fomcminutes20250319.pdf::chunk0000,fomcminutes20250319.pdf,0.146134,0.248051


LLM stage completed in 368.69s


,attempt,status,latency_s,context_chars,num_predict
0,1,parse_failed,187.34,2800,320
1,2,parse_failed,104.79,1959,240
2,3,parse_failed,76.53,1500,180


{
"generated_at_utc": "{timestamp}",
"executive_summary": "US macro outlook: moderate inflation pressures, stable unemployment rate, and gradual interest rate hikes.",
"regime_call": {
"growth_momentum": "stable",
"inflation_trend": "moderate",
"policy_bias": "gradual"
  ,"recession_risk": "low"
  ,"confidence": ".5"
},
"topic_signals": [
{
"topic": "inflation",
"view": "moderate",
"why_it_matters": "higher term premiums and perceived shift in balance of risks"
  ,"confidence": ".6"
,"evidence": ["fomcminutes20241107.pdf::chunk0011", "20250207_mprfullreport.pdf::chunk0175"]
},
{
"topic": "unemployment",
"view": "stable
[check] JSON parse failed after retries


## 5) Save artifacts

In [6]:
run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")

hits_path = OUTPUT_DIR / f"hits_{run_ts}.csv"
hits_df.to_csv(hits_path, index=False)
print("Saved:", hits_path)

txt_path = OUTPUT_DIR / f"macro_answer_{run_ts}.txt"
txt_path.write_text(llm_text)
print("Saved:", txt_path)

if parsed is not None:
    json_path = OUTPUT_DIR / f"macro_answer_{run_ts}.json"
    with open(json_path, "w") as f:
        json.dump(parsed, f, indent=2)
    print("Saved:", json_path)


Saved: /Users/atheeshkrishnan/AK/DEV/hawkdove/fed_macro_mvp/outputs/hits_20260315_203448.csv
Saved: /Users/atheeshkrishnan/AK/DEV/hawkdove/fed_macro_mvp/outputs/macro_answer_20260315_203448.txt
